In [35]:
import numpy as np
import pandas as pd

In [36]:
# makes sure we are using proper data types
"""
Input: 
    df : pd.DataFrame
    uses the league_data.csv

Returns:
    pd.DataFrame
    A copy of the DataFrame with normalized types
"""
def validate_data(df):
    df = df.copy()
    
    df["season"] = df["season"].astype(int)
    df["date"] = pd.to_datetime(df["date"])
    df["score1"] = df["score1"].astype(int)
    df["score2"] = df["score2"].astype(int)
    df["result"] = df["result"].astype(int)
    return df

In [37]:

"""
Input: 
    df (pd.DataFrame)
        uses the validated league_data.csv
    league (str)
        Identifies which league is being used

Returns:
    pd.DataFrame
        Has two row pers game, one per each team
        (league, season, team_id, points_for, points_against, result, is_home)
        result if from THIS team perspective: 3 = win, 0 = loss, 1 = tie
        is_home: 1 for home team row, 0 for away team row

How it works:
    The code splits each game into a home row and an away row.
    The input "result" is always from the home team perspective. The aways team's "result" is remapped
    This creates a dataset that is useful for season aggregation.
"""
def create_team_rows(df, league="NFL"):

    home = pd.DataFrame({
        "league": league,
        "season": df["season"],
        "team_id": df["team1"],
        "points_for": df["score1"],
        "points_against": df["score2"],
        "result": df["result"],
        "is_home": 1  
    })

    result_from_away_perspective = {3: 0, 0: 3, 1: 1}
    away = pd.DataFrame({
        "league": league,
        "season": df["season"],
        "team_id": df["team2"],
        "points_for": df["score2"],
        "points_against": df["score1"],
        "result": df["result"].map(result_from_away_perspective),
        "is_home": 0 
    })

    return pd.concat([home, away], ignore_index = True)

In [38]:
"""
Input: 
    rows (pd.DataFrame)
        uses the output of 'create_team_rows' function

Returns:
    pd.DataFrame
        End of season standinger per league, season, team_id

How it works:
    The code caluclates the number of win/losses/ties from 'result',
    We aggregate by (league, season, team_id) to the season totals,
    The win percent and point diffrentions are then calculated,
    We sort by win_pct (DESC), point_diff (DESC), team_id (ASC),
    Ranks are then assigned, 1 being the best.
"""
def get_standing(rows):

    team_rows = rows.copy()
    team_rows["wins"] = (team_rows["result"] == 3).astype(int)
    team_rows["losses"] = (team_rows["result"] == 0).astype(int)
    team_rows["ties"] = (team_rows["result"] == 1).astype(int)

    standings = team_rows.groupby(["league", "season", "team_id"], as_index=False).agg({
        "result": "count",
        "wins": "sum",
        "losses": "sum", 
        "ties": "sum",
        "points_for": "sum",
        "points_against": "sum"
    }).rename(columns={"result": "games_played"})

    standings["win_pct"] = (standings["wins"] + 0.5 * standings["ties"]) / standings["games_played"]
    standings["point_diff"] = standings["points_for"] - standings["points_against"]
    standings["win_pct"] = standings["win_pct"].round(3)
    standings = standings.sort_values(["league", "season", "win_pct", "point_diff", "team_id"],
        ascending=[True, True, False, False, True])
    standings["rank"] = standings.groupby(["league", "season"]).cumcount() + 1

    columns = ["league", "season", "team_id", "games_played", "wins", "losses", 
              "ties", "win_pct", "points_for", "points_against", "point_diff", "rank"]
    
    return standings[columns]

In [39]:
"""
Input: 
    standings (pd.DataFrame)
        Output of 'get_standing()'
    output (str, "data/us_leagues/csv")
        The directory of where the CSV will be written
    filename (str)
        Name of the CSV file to create

Returns:
    str
        The full path to the written CSV

How it works:
    It sorts the standings by season and rank for readability, and build the output path
    The code write the CSV file and prints out the path and number of rows.
"""
def get_rankings(standings, output = "data/us_leagues/csv", filename="nfl_standings.csv"):
    path = f"{output}/{filename}"
    standings_sorted = standings.sort_values(["season", "rank"])
    standings_sorted.to_csv(path, index=False)
    seasons = standings["season"].unique()
    total_teams = len(standings)
    
    print(f"Exported: {path}")
    print(f"Total rows: {total_teams}")
    
    return path


In [40]:
# run to generate nfl_standings.csv
nfl_data = pd.read_csv("../csv/nfl_data.csv")
nfl_clean = validate_data(nfl_data)
team_rows = create_team_rows(nfl_clean, league="NFL")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "end_of_season_nfl.csv")

Exported: ../csv/end_of_season_nfl.csv
Total rows: 1738


In [41]:
# run to generate mlb_standings.csv
mlb_data = pd.read_csv("../csv/mlb_data.csv")
mlb_clean = validate_data(mlb_data)
team_rows = create_team_rows(mlb_clean, league="MLB")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "end_of_season_mlb.csv")

Exported: ../csv/end_of_season_mlb.csv
Total rows: 1288


In [42]:
# run to generate nba_standings.csv
nba_data = pd.read_csv("../csv/nba_data.csv")
nba_clean = validate_data(nba_data)
team_rows = create_team_rows(nba_clean, league="NBA")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "end_of_season_nba.csv")

Exported: ../csv/end_of_season_nba.csv
Total rows: 1662
